In [ ]:
!pip install pymanopt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.1 MB/s eta 0:00:00


In [ ]:
%reset -f

In [ ]:
import numpy as np
# import tensorflow as tf
import cupy as cp
# import jax.numpy as jnp
# import jax
# from jax import jit, lax
# from jax import device_put
# from jax.lib import xla_bridge

# import pymanopt
# from pymanopt.manifolds import Oblique
# from pymanopt.optimizers import TrustRegions
#from cupy.dtypes import float8_e4m3fn, float8_e5m2

import time
import gc

In [ ]:
import os

# 2. Use CUB and cuTENSOR backends (Faster reductions/elementwise)
os.environ["CUPY_ACCELERATORS"] = "cub,cutensor"

# 3. Tune the Compiler
# Tells the GPU compiler to optimize for the A100 architecture specifically
os.environ["CUDA_CACHE_MAXSIZE"] = "2147483648" # 2GB Cache
# os.environ["CUDA_CACHE_MAXSIZE"] = "4294967296" # 2GB Cache

# # TF32 settings
# # Allow CuPy/CUDA to use Tensor Cores for Float32
# os.environ["NVIDIA_TF32_OVERRIDE"] = "1"

# # Tell CuPy to allow TF32
# cp.cuda.set_allocator(cp.cuda.MemoryPool().malloc)
# # Note: CuPy enables TF32 by default on A100,
# # but checking specific matrix math settings helps.

In [ ]:
def reset_memory():
    # Delete all possible GPU arrays
    globals_ = list(globals().keys())
    for var in globals_:
        if var in ['cp', 'jnp', 'reset_memory']: continue
        if isinstance(globals()[var], (cp.ndarray, jnp.ndarray)):
            del globals()[var]

    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()

In [ ]:
n = 30000
k = int(np.ceil((n * 2) ** 0.5 ))
shift = 50/n

C = cp.random.randn(n,n)/n
C = C + C.T
cp.fill_diagonal(C, C.diagonal() + shift)

# generate_B.seed = 0
# B = generate_B.randn(k,n)
B = cp.random.randn(k,n)
B = B/cp.linalg.norm(B,axis = 0)

# cp_f8 = float8_e5m2
# cp_f8 = float8_e4m3fn

C_16 = cp.array(C,dtype = cp.float16)
B_16 = cp.array(B,dtype = cp.float16)

# C_16 = cp.array(C,dtype = cp.float32)
# B_16 = cp.array(B,dtype = cp.float32)

B_16.dot(C_16)
cp.cuda.Device().synchronize()
# 1000 is good enough do not exceed 1500
K16 = [100,100,100,100,100,100,100,100,100,100]
for i in range(len(K16)):
  cg0 = time.time()
  for t in range(K16[i]):
      B_16 = B_16.dot(C_16)
      B_16 = B_16/cp.linalg.norm(B_16,axis = 0)
  cp.cuda.Device().synchronize()
  cg1 = time.time()
  obj64 = cp.tensordot(B_16,B_16.dot(C))
  print("block time",cg1-cg0)
  print("obj 64",obj64)

block time 0.29315948486328125
obj 64 531.7354455454063
block time 0.17740273475646973
obj 64 532.266900628419
block time 0.17747712135314941
obj 64 532.348577182202
block time 0.17728161811828613
obj 64 532.3581050493909
block time 0.17742300033569336
obj 64 532.3482192589628
block time 0.17725920677185059
obj 64 532.3666229534808
block time 0.17734646797180176
obj 64 532.4365660939056
block time 0.17740535736083984
obj 64 532.44631151038
block time 0.1760542392730713
obj 64 532.4124342630134
block time 0.17621850967407227
obj 64 532.3564578663279


In [ ]:
n = 30000
k = int(np.ceil((n * 2) ** 0.5 ))

repeat = 50

f_cg = np.zeros((repeat,10))
t_cg = np.zeros((repeat,10))

shift = 50/n

cp.random.seed(0)
for rep in range(repeat):
  print("----------------REP++",rep,"++---------------------------------------")
  # reset_memory()
  C = cp.random.randn(n,n)/n
  C = C + C.T
  cp.fill_diagonal(C, C.diagonal() + shift)

  # generate_B.seed = 0
  # B = generate_B.randn(k,n)
  B = cp.random.randn(k,n)
  B = B/cp.linalg.norm(B,axis = 0)

  C_16 = cp.array(C,dtype = cp.float16)
  B_16 = cp.array(B,dtype = cp.float16)
  dummy_res = B_16.dot(C_16)
  cp.cuda.Device().synchronize()

  # 1000 is good enough do not exceed 1500
  K16 = [100,100,100,100,100,100,100,100,100,100]
  for i in range(len(K16)):
    cg0 = time.time()
    for t in range(K16[i]):
        B_16 = B_16.dot(C_16)
        B_16 = B_16/cp.linalg.norm(B_16,axis = 0)
    cp.cuda.Device().synchronize()
    cg1 = time.time()

    # obj64 = cp.tensordot(B_16,B_16.dot(C))-shift *n
    obj64 = cp.tensordot(B_16,B_16.dot(C))
    print("block time",cg1-cg0)
    print("obj 64",obj64)
    f_cg[rep,i] = obj64
    t_cg[rep,i] = cg1-cg0

----------------REP++ 0 ++---------------------------------------
block time 0.17626214027404785
obj 64 531.7189052264763
block time 0.17584538459777832
obj 64 532.2649036639024
block time 0.1718306541442871
obj 64 532.3354203126521
block time 0.17187833786010742
obj 64 532.3419882778431
block time 0.17162537574768066
obj 64 532.419859955777
block time 0.17156362533569336
obj 64 532.4278592750093
block time 0.17165231704711914
obj 64 532.4081220871833
block time 0.17160916328430176
obj 64 532.4078006281985
block time 0.16938185691833496
obj 64 532.4243267075132
block time 0.16979503631591797
obj 64 532.4763499022883
----------------REP++ 1 ++---------------------------------------
block time 0.17077279090881348
obj 64 531.7614723455501
block time 0.17058348655700684
obj 64 532.3140128755947
block time 0.16880011558532715
obj 64 532.3895403014786
block time 0.16885852813720703
obj 64 532.3959609370896
block time 0.16881418228149414
obj 64 532.4779515125921
block time 0.1689772605895996


KeyboardInterrupt: 

In [ ]:
!mkdir -p /content/drive/MyDrive/result_folder
np.savez("/content/drive/MyDrive/result_folder/results_gpu_float16_final_h100.npz",f_cg = f_cg, t_cg = t_cg)

In [ ]:
np.cumsum(t_cg.mean(axis = 0))

In [ ]:
f_cg.mean(axis = 0)

In [ ]:
print(cp.__version__)

In [ ]:
!nvidia-smi